<a href="https://colab.research.google.com/github/HR-Supaero/deep-learning/blob/main/demo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 Retrieval-Augmented Generation (RAG) Demo Notebook

Welcome! This notebook demonstrates **RAG systems** step by step:
- 📄 Document ingestion  
- 🧠 Embedding & indexing  
- 🔍 Retrieval  
- 🤖 LLM-based generation  



# Retrieval Augmented Generation

[Faiss documentation](https://faiss.ai/index.html)

In [1]:
# Run those install if you are using colab
!pip install faiss-gpu-cu12
!pip install -q transformers sentence-transformers pypdf
!pip install evaluate rouge_score
!pip install -qU ragas langchain-google-genai langchain-community datasets
!pip install langchain_qdrant
!pip install fastembed-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0c0f3356dde9125e819eab569d6a2ac53e34cee1e9d784926cd1a06198494712
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Imports
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
from tqdm.notebook import tqdm
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
import evaluate

# 🍼 Toy example

In [3]:
# Parameters
chunk_size = 50
k = 2             # Top k docs

In [4]:
documents = [
    "Ernest Hemingway was born in Oak Park, Illinois, in 1899. He was an American novelist, short-story writer, and journalist.",
    "The middle ear is the portion of the ear internal to the eardrum, and external to the oval window of the inner ear.",
    "The Sun Also Rises is a 1926 novel by American writer Ernest Hemingway that involves a group of American and British expatriates.",
    "The three ossicles of the middle ear are the malleus, incus, and stapes, which transfer sound vibrations."
]

⚗ Embedding Model

In [5]:
# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🖌 Generation model

In [6]:
gen_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
generator = AutoModelForCausalLM.from_pretrained(gen_model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🛠 First architecture

In [7]:
# Divide the docs into chunks
def chunker_v0(text, chunk_size):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

all_chunks = []
for doc in documents:
    all_chunks.extend(chunker_v0(doc, chunk_size))

print(f"Total chunks created: {len(all_chunks)}")
print(all_chunks[0])

Total chunks created: 4
Ernest Hemingway was born in Oak Park, Illinois, in 1899. He was an American novelist, short-story writer, and journalist.


In [8]:
chunk_embeddings = embed_model.encode(all_chunks)
print(chunk_embeddings.shape)

# Create FAISS Index
dimension = chunk_embeddings.shape[1]
print(f"Latent space dim: {dimension}")
index = faiss.IndexFlatIP(dimension) # Inner Product (Cosine Similarity)
index.add(np.array(chunk_embeddings).astype('float32'))
faiss.write_index(index, "my_vector_store.index")
print("Vector store built and saved.")


(4, 384)
Latent space dim: 384
Vector store built and saved.


###◀ Retrieval

In [9]:
def retriever_v0(query, k=k):   # Will give us the best k documents
    query_vec = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k)
    return [all_chunks[i] for i in indices[0]]

print(f"🔍 Searching for: Hemingway")
context = retriever_v0("Ernest Hemingway")
print(f"📖 Best {k} documents found: {context}")

🔍 Searching for: Hemingway
📖 Best 2 documents found: ['Ernest Hemingway was born in Oak Park, Illinois, in 1899. He was an American novelist, short-story writer, and journalist.', 'The Sun Also Rises is a 1926 novel by American writer Ernest Hemingway that involves a group of American and British expatriates.']


## Generation

In [10]:
def generate_answer(query, context):
    # Instruction-style prompt
    prompt = (
        f"You are a helpful assistant. Use the context below to answer the question.\n"
        f"Context: {' '.join(context)}\n"
        f"Question: {query}\n"
        f"Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = generator.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    return full_text.split("Answer:")[-1].strip()

In [11]:
def run_rag_dummy(user_query):
    print(f"🔍 Searching for: {user_query}")
    context = retriever_v0(user_query)
    print(f"📖 Context found: {context}")

    answer = generate_answer(user_query, context)
    return answer

# Test 1: Hemingway
print(f"Final Answer: {run_rag_dummy('Where was Hemingway born?')}")

print("-" * 30)

# Test 2: Biology
print(f"Final Answer: {run_rag_dummy('What are the bones in the middle ear?')}")

print("-" * 30)

# Test 3: Novel
print(f"Final Answer: {run_rag_dummy('When was The Sun Also Rises written ?')}")

print("-" * 30)

# Test 4: Random question
print(f"Final Answer: {run_rag_dummy('Who is the GOAT ? Messi Or Ronaldo ?')}")

🔍 Searching for: Where was Hemingway born?
📖 Context found: ['Ernest Hemingway was born in Oak Park, Illinois, in 1899. He was an American novelist, short-story writer, and journalist.', 'The Sun Also Rises is a 1926 novel by American writer Ernest Hemingway that involves a group of American and British expatriates.']
Final Answer: Ernest Hemingway was born in Oak Park, Illinois, in 1899. He was an American novelist, short-story writer, and journalist
------------------------------
🔍 Searching for: What are the bones in the middle ear?
📖 Context found: ['The middle ear is the portion of the ear internal to the eardrum, and external to the oval window of the inner ear.', 'The three ossicles of the middle ear are the malleus, incus, and stapes, which transfer sound vibrations.']
Final Answer: The bones of the middle ear are the bones of the middle ear. The bones of the middle ear are the bones of the middle ear. The bones of the middle ear are the bones of the middle ear. The bones of th

Ok two observations:

While the first question is anwered correctly it seems the LLM just re gave us the context, while not wrong per se, it seems it was helped by the chunk size
It is like answering a question by not reformulating anything

The second answer is wrong though and it is perhaps a confirmation of what we suspected from the first response. The RAG doesnt understand the fact that "bones" and "Ossciles" are semantically similar.Worse, the second document is the best one for answering the question !

We observe what is called collapse, where instead of not returning an answer, the LLM still return a context


First of all lets try another chunker, since ours is pretty bruttal

##📚 Bigger example: Wikipedia questions

https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia

In [12]:
# Recursive chunker to preserve semantic sense

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Prioritize keeping paragraphs together, then sentences, then words.
    chunk_size=25,  # Aim for chunks of around 256 characters
    chunk_overlap=24,  # Overlap chunks by 16 characters to preserve context
)

def chunker_v1(documents):
  chunks = []
  for doc in documents:
      chunks.extend(text_splitter.split_text(doc))
  return chunks

chunks = chunker_v1(documents)
chunks[0]

'Ernest Hemingway was born'

Test on bigger data set

In [13]:
df_QnA = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet")


In [14]:
df_QnA.head()

,question,answer
id,,
0,Was Abraham Lincoln the sixteenth President of...,yes
2,Did Lincoln sign the National Banking Act of 1...,yes
4,Did his mother die of pneumonia?,no
6,How many long was Lincoln's formal education?,18 months
8,When did Lincoln begin his political career?,1832


In [15]:
df_docs = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet")
df_docs.head()

,passage
id,
0,"Uruguay (official full name in ; pron. , Eas..."
1,"It is bordered by Brazil to the north, by Arge..."
2,Montevideo was founded by the Spanish in the e...
3,The economy is largely based in agriculture (m...
4,"According to Transparency International, Urugu..."


In [16]:
documents = df_docs['passage']
print(documents.shape)

(3200,)


In [17]:
questions = df_QnA['question']
print(questions.shape)

(918,)


In [18]:
answers_expected = df_QnA['answer']
print(answers_expected.shape)

(918,)


In [19]:
all_chunks = []
for doc in documents:
    all_chunks.extend(chunker_v0(doc, chunk_size))

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 5781


In [20]:
# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(all_chunks)

# Create FAISS Index
dimension = chunk_embeddings.shape[1]
print(f"Latent space dim: {dimension}")
index = faiss.IndexFlatIP(dimension) # Inner Product (Cosine Similarity)
index.add(np.array(chunk_embeddings).astype('float32'))

# Functional Tip: Save your index to disk
faiss.write_index(index, "my_vector_store.index")
print("Vector store built and saved.")

Latent space dim: 384
Vector store built and saved.


In [21]:
# Your existing code for RAG
import random
n_questions = 5     # Can be very long to execute

rag_answer = []
answers_rag = []


for q in tqdm(questions[0:n_questions], desc="Processing questions with RAG"):
    answers_rag.append(run_rag_dummy(q))

# Print a randomly selected response
print(answers_rag[random.randint(0, n_questions - 1)])

Processing questions with RAG:   0%|          | 0/5 [00:00<?, ?it/s]

🔍 Searching for: Was Abraham Lincoln the sixteenth President of the United States?
📖 Context found: ['Abraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would', 'Young Abraham Lincoln']
🔍 Searching for: Did Lincoln sign the National Banking Act of 1863?
📖 Context found: ['legislation involved economic matters, including the first income tax and higher tariffs. Also included was the creation of the system of national banks by the National Banking Acts of 1863, 1864, and 1865, which allowed the creation of a strong national financial system. Congress created and Lincoln approved the Department', 'Lincoln believed in the Whig theory of the presidency, which left Congress to write the laws while he signed them, vetoing only t

In [22]:
# Extract only the answer (first line or before \n)
extracted_answers = []
context_retrieved = []
for answer in answers_rag:
    # Split by newline and take the first part
    extracted = answer.split('\n')[0]
    context_retrieved.append(answer.split('\n')[1:])
    extracted_answers.append(extracted)
    context_retrieved.append(answer.split('\n')[1:])

print(len(extracted_answers))
print(extracted_answers)

5
['Yes, he was.', 'No. Lincoln signed the National Banking Act of 1863, which created the National Banking Act of 1863, which created the National Banking Act of 1863, which created the National Banking Act of 1863, which created the National Banking Act of 1863, which created the', 'No.', "Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, once walking. just to borrow one", 'Lincoln was born in 1832, in the town of Sangamon, Illinois. He was raised in the town of Lincoln, Illinois, and was educated at the University of Illinois. He was a member of the Illinois State Legislature, and was elected to']


In [23]:
top_k_context = []
for i in range(len(context_retrieved)):
  if (i%k == 0):
    top_k_context.append(context_retrieved[i])

print(len(top_k_context))
print(top_k_context)

5
[['Context: Abraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken'], [], ['Question:'], ["Context: Lincoln's formal education consisted of about 18 months of schooling"], []]


In [24]:


# Ensure that the lengths match for creating the DataFrame
# questions, answers_expected, and extracted_answers are Series/lists, so slicing to match the processed length
eval_df = pd.DataFrame({
    'question': questions[0:len(extracted_answers)].values,
    'expected_answer': answers_expected[0:len(extracted_answers)].values,
    'context retrieved': top_k_context,
    'rag_answer': extracted_answers
})
display(eval_df.head())


,question,expected_answer,context retrieved,rag_answer
0,Was Abraham Lincoln the sixteenth President of...,yes,"[Context: Abraham Lincoln (February 12, 1809 â...","Yes, he was."
1,Did Lincoln sign the National Banking Act of 1...,yes,[],No. Lincoln signed the National Banking Act of...
2,Did his mother die of pneumonia?,no,[Question:],No.
3,How many long was Lincoln's formal education?,18 months,[Context: Lincoln's formal education consisted...,Lincoln's formal education consisted of about ...
4,When did Lincoln begin his political career?,1832,[],"Lincoln was born in 1832, in the town of Sanga..."


### RAG Quality Metrics




For a more nuanced evaluation of RAG quality, especially considering the semantic similarity and correctness of generated text, you can use metrics beyond simple exact matching:

*   **ROUGE-L**: These metrics compare the generated answer to the reference answer based on overlapping words or sequences. ROUGE-L (Longest Common Subsequence) is particularly good for assessing the main ideas.
*   **Embedding Similarity**: Mesuring the rag answer embedding angle to the expected response

Libraries like `evaluate` (Hugging Face) provide implementations for ROUGE and other generation metrics, which can be very helpful for a more comprehensive evaluation.

In [25]:

rouge = evaluate.load('rouge')

In [26]:
rouge_score = rouge.compute(predictions=extracted_answers, references=answers_expected[0:n_questions])
print(f"Rouge Score {rouge_score}")

Rouge Score {'rouge1': np.float64(0.3190476190476191), 'rouge2': np.float64(0.013333333333333332), 'rougeL': np.float64(0.33452380952380956), 'rougeLsum': np.float64(0.33452380952380956)}


Comparasion with SOTA rouge scores

| Dataset | ROUGE-1 (Unigrams) | ROUGE-2 (Bigrams) | ROUGE-L (Longest Sequence) |
|---|---|---|---|
| CNN / Daily Mail | 44.0 – 47.5 | 20.0 – 22.0 | 41.0 – 44.5 |
| XSum (Extreme) | 45.0 – 48.0 | 22.0 – 25.0 | 37.0 – 40.0 |
| SAMSum (Dialogue) | 52.0 – 55.0 | 28.0 – 31.0 | 49.0 – 52.0 |
| Multi-News | 48.0 – 50.0 | 19.0 – 21.0 | 44.0 – 46.0 |

In [27]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

emb1 = model.encode("The sun is a star.")
emb2 = model.encode("Our solar system's center is a stellar body.")

cos_sim = util.cos_sim(emb1, emb2)
print(f"Semantic Similarity: {cos_sim.item():.4f}")

Semantic Similarity: 0.6063


In [28]:
avg_latent_angle = []
for rag, expected in zip(extracted_answers, answers_expected[0:n_questions]):
  emb1 = model.encode(rag)
  emb2 = model.encode(expected)
  cos_sim = util.cos_sim(emb1, emb2)
  avg_latent_angle.append(cos_sim.item())

avg_latent_angle = sum(avg_latent_angle) / len(avg_latent_angle)
print(f"Semantic Similarity: {avg_latent_angle:.4f}")


Semantic Similarity: 0.3123


#❓ What could be improved ?

* Try with bigger LLM models
* LLM based evaluation
* Play around the prompt template
